# CenterLoc3D on SVLD-3D — Colab training

Runtime yang dipakai: **L4 GPU** (T4 juga bisa, lebih lambat).
Cek dulu: *Runtime > Change runtime type > L4*.

Urutan: sel **Setup** → sel **Smoke** (~10 menit) → kalau lolos, sel **Full**.

Smoke menulis ke `/content/out` (tidak mengotori Drive) dan selalu mulai
dari awal. Full menulis ke `/content/drive/MyDrive/centerloc3d` dan
otomatis resume dari `checkpoint_last.pth` kalau sesi putus.

## 1. Setup (mount Drive, cek GPU)

In [ ]:
# Opsional tapi disarankan: checkpoint selamat kalau sesi putus.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 2. Smoke test (~10 menit)

Data A-C (3,4 GB) + 200 gambar + 2 epoch, hasil ke `/content/out`.
Tujuannya memastikan unduh, ekstrak, txt, model, loss, dan checkpoint
jalan — bukan hasil. Smoke tidak pernah memakai checkpoint lama.

Sel ini selalu mengambil kode terbaru dari repo, jadi aman dijalankan
ulang setelah ada perbaikan skrip.

In [ ]:
!rm -rf /content/carom
!git clone -q --depth 1 https://github.com/themasfebrianto/carom-research.git /content/carom
!cd /content/carom && python scripts/colab_train_centerloc3d.py --preset smoke

## 3. Full training

Semua arsip (13,5 GB, 14.592 gambar), 50 epoch, batch 8.

- Perkiraan: L4 ~9-14 jam, T4 ~20-26 jam (dari waktu per iterasi smoke).
- Kalau sesi putus: jalankan ulang sel ini, resume otomatis.
- Kalau kuota mepet: `--epochs 30`. Resume tetap jalan karena tag run
  tidak menyertakan jumlah epoch.
- Unduh 13,5 GB hanya ~3 menit di Colab, jadi cache tar di Drive
  opsional: tambah `--cache-tar /content/drive/MyDrive/centerloc3d/svld_data.tar`.

In [ ]:
!rm -rf /content/carom
!git clone -q --depth 1 https://github.com/themasfebrianto/carom-research.git /content/carom
!cd /content/carom && python scripts/colab_train_centerloc3d.py --preset full

## 4. Cek progres

Log ada di `train_log.csv` dan `checkpoint_last.pth` di folder output.

In [ ]:
import pandas as pd
from pathlib import Path

out = Path('/content/drive/MyDrive/centerloc3d')
if not out.exists():
    out = Path('/content/out')
print(sorted(p.name for p in out.glob('*')))
csv = out / 'train_log.csv'
if csv.exists():
    df = pd.read_csv(csv)
    display(df.tail(10))
    display(df[['train_loss', 'val_loss']].plot())